<a href="https://colab.research.google.com/github/TrSaleMane/municipality-ai-course/blob/main/docs/day2_rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Day 2：RAG実装と住民窓口チャットボット構築
## 新人SE向け 自治体AIシステム開発 模擬体験講座

---

### このノートブックの目標

1. RAG（Retrieval-Augmented Generation）の仕組みを理解する
2. PDFから知識を取り込んでベクトルDBに登録できる
3. 自治体FAQ文書を読んで正確に回答するAIを作る
4. GradioでチャットUIを構築して動作確認する

### RAGとは？

```
❌ 普通のLLM
  質問 → LLM（学習済み知識だけで回答）→ 古い情報・ハルシネーションのリスク

✅ RAG（Retrieval-Augmented Generation）
  質問 → [関連文書を検索] → [文書 + 質問をLLMに渡す] → 正確な回答
         ↑ここがRAGの核心
```

> 📌 **源内OSS** では Amazon OpenSearch + Bedrock でRAGを実現しています。  
> 本講座では ChromaDB + Gemini API で同等の仕組みを体験します。  
> Copyright © デジタル庁 / Licensed under MIT License

---
## RAGシステムのアーキテクチャ

```
【事前準備フェーズ（インデックス構築）】

  FAQ文書(PDF)
      ↓ テキスト抽出
  テキスト
      ↓ チャンク分割（500文字ずつ等）
  チャンク群
      ↓ Embeddingモデルでベクトル化
  ベクトルDB（ChromaDB）に保存


【質問応答フェーズ（推論）】

  住民の質問
      ↓ 質問もベクトル化
  ChromaDBで類似検索 → 関連チャンクを取得
      ↓
  [関連チャンク + 質問] をプロンプトに組み込む
      ↓
  Gemini API が回答生成
      ↓
  住民に回答
```

---
## Section 1：環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q google-generativeai chromadb pymupdf gradio langchain langchain-google-genai

In [ ]:
import os
import fitz          # PyMuPDF：PDF読み取り
import chromadb      # ベクトルDB
import gradio as gr  # チャットUI
import google.generativeai as genai
from google.colab import userdata
from typing import List, Tuple

# APIキー設定
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

print('✅ ライブラリ読み込み完了')

---
## Section 2：演習用FAQデータの準備

実際の自治体FAQを模したサンプルデータを使います。  
（本番では実際の自治体公開PDFを使用してください）

In [ ]:
# ======================================================
# 演習用：自治体FAQ サンプルデータ
# 実際の講座では自治体公開PDFに差し替えてください
# ======================================================

SAMPLE_FAQ = """
=== 住民票・戸籍 ===

Q: 住民票の写しを取得するにはどうすればいいですか？
A: 住民票の写しは以下の方法で取得できます。
   【窓口】市役所1階の市民課窓口（平日8:30〜17:15）。本人確認書類と手数料300円が必要です。
   【コンビニ】マイナンバーカードとコンビニのマルチコピー機を使って取得できます。手数料は200円です。
   【郵便】申請書と本人確認書類のコピー、返信用封筒、手数料分の定額小為替を同封して郵送してください。

Q: 住民票の転入届はどこで手続きしますか？
A: 引越し後14日以内に市役所市民課窓口で手続きしてください。
   必要なもの：転出証明書（前の市区町村で発行）、本人確認書類、マイナンバーカード（お持ちの方）
   手続きは平日8:30〜17:15です。第2・第4土曜日は9:00〜12:00も受け付けています。

Q: 戸籍謄本はどこで取得できますか？
A: 本籍地の市区町村窓口で取得できます。本市に本籍がある方は市民課窓口（1階）でお手続きください。
   手数料：戸籍謄本 750円、戸籍抄本 750円
   郵便請求も可能です。詳しくは市のホームページをご確認ください。

=== 国民健康保険 ===

Q: 国民健康保険に加入するにはどうすればいいですか？
A: 会社を退職した日・扶養から外れた日など、健康保険の資格を失った日から14日以内に手続きが必要です。
   手続き窓口：市役所2階 保険年金課
   必要なもの：健康保険資格喪失証明書、本人確認書類、マイナンバーカードまたは通知カード

Q: 国民健康保険料の減免制度はありますか？
A: 収入が激減した場合や災害にあった場合など、一定の条件のもとで保険料の減免が受けられる場合があります。
   詳しくは保険年金課（電話：000-0000-0000）までお問い合わせください。

=== 介護保険 ===

Q: 介護保険のサービスを利用するにはどうすればいいですか？
A: まず要介護認定の申請が必要です。
   ①市役所3階 介護福祉課または地域包括支援センターへ申請
   ②認定調査員が自宅を訪問して調査
   ③医師の意見書をもとに介護認定審査会で審査
   ④結果通知（申請から約30日）
   認定後、ケアマネジャーと相談してサービスを利用開始できます。

=== 道路・土木 ===

Q: 道路の穴や破損を見つけたらどこに連絡すればいいですか？
A: 市が管理する道路の場合は、建設課道路維持係（電話：000-0000-0001）までご連絡ください。
   連絡の際は、場所（住所・目標物）と損傷の状況をお知らせください。
   県道・国道の場合は、それぞれ県土木事務所・国道事務所が管轄となります。

Q: 道路工事の許可申請はどこに提出しますか？
A: 道路占用許可申請は建設課道路管理係に提出してください。
   工事着手の2週間前までに申請書・添付書類を提出する必要があります。
   詳しくは建設課（電話：000-0000-0002）にお問い合わせください。

=== 子育て・保育 ===

Q: 保育所に入所申請するにはどうすればいいですか？
A: 毎年10月〜11月頃に翌年4月入所の申し込みを受け付けます。
   窓口：こども課（市役所3階）または各保育所
   必要書類：入所申請書、就労証明書（保護者双方分）、マイナンバー書類等
   選考結果は翌年2月頃にお知らせします。

Q: 児童手当を受け取るにはどうすればいいですか？
A: 出生・転入などの事由が発生した日の翌日から15日以内にこども課で申請してください。
   支給額：3歳未満 月15,000円、3歳〜小学校修了前 月10,000円（第3子以降15,000円）、中学生 月10,000円
   ※所得制限があります。詳しくはこども課にお問い合わせください。
"""

print(f'✅ FAQデータ準備完了（{len(SAMPLE_FAQ)}文字）')

---
## Section 3：テキストのチャンク分割

長い文書をそのままベクトル化するのではなく、**意味のある単位に分割（チャンク化）**します。

In [ ]:
# ======================================================
# 【演習 3-1】テキストをチャンクに分割する
# ======================================================

def split_into_chunks(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    テキストを指定サイズのチャンクに分割する
    
    Args:
        text: 分割するテキスト
        chunk_size: チャンクの最大文字数
        overlap: チャンク間のオーバーラップ文字数（文脈の連続性を保つため）
    """
    # Q&Aの境界で分割する（より意味のある分割）
    qa_pairs = [block.strip() for block in text.split('\n\n') if block.strip()]
    
    chunks = []
    for qa in qa_pairs:
        if len(qa) <= chunk_size:
            chunks.append(qa)
        else:
            # 長いテキストはさらに分割
            for i in range(0, len(qa), chunk_size - overlap):
                chunk = qa[i:i + chunk_size]
                if chunk.strip():
                    chunks.append(chunk)
    
    # 空チャンクと見出し行のみのチャンクを除外
    chunks = [c for c in chunks if len(c) > 20]
    return chunks


chunks = split_into_chunks(SAMPLE_FAQ)
print(f'✅ チャンク数：{len(chunks)}個')
print('\n--- チャンク例（最初の2つ）---')
for i, chunk in enumerate(chunks[:2]):
    print(f'\n[チャンク {i+1}]')
    print(chunk)

---
## Section 4：ベクトルDBへの登録

**Embedding（埋め込み）** とは、テキストを数値のベクトルに変換することです。  
意味が似ているテキストは、ベクトル空間でも近い位置に配置されます。

In [ ]:
# ======================================================
# 【演習 4-1】ChromaDBにチャンクを登録する
# ======================================================

class FAQVectorStore:
    """FAQ文書のベクトルストア"""

    def __init__(self, collection_name: str = 'municipality_faq'):
        # ChromaDB クライアント初期化（インメモリ）
        self.client = chromadb.Client()
        # コレクション（テーブルのようなもの）を作成
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={'description': '自治体FAQベクトルストア'}
        )

    def embed_text(self, text: str) -> List[float]:
        """Gemini Embedding APIでテキストをベクトル化"""
        result = genai.embed_content(
            model='models/text-embedding-004',
            content=text,
            task_type='retrieval_document'
        )
        return result['embedding']

    def add_documents(self, chunks: List[str]):
        """チャンクをベクトル化してDBに登録"""
        print(f'🔄 {len(chunks)}個のチャンクをベクトル化中...')
        embeddings = [self.embed_text(chunk) for chunk in chunks]
        
        self.collection.add(
            documents=chunks,
            embeddings=embeddings,
            ids=[f'chunk_{i}' for i in range(len(chunks))]
        )
        print(f'✅ {len(chunks)}個のチャンクをDBに登録完了')

    def search(self, query: str, n_results: int = 3) -> List[str]:
        """クエリに類似したチャンクを検索"""
        query_embedding = genai.embed_content(
            model='models/text-embedding-004',
            content=query,
            task_type='retrieval_query'
        )['embedding']

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]  # 類似チャンクのリストを返す


# ベクトルストアを構築
vector_store = FAQVectorStore()
vector_store.add_documents(chunks)

# 検索テスト
print('\n--- 検索テスト ---')
test_query = '住民票をコンビニで取りたいです'
results = vector_store.search(test_query, n_results=2)
print(f'質問: {test_query}')
for i, result in enumerate(results, 1):
    print(f'\n[検索結果 {i}]\n{result}')

---
## Section 5：RAGチェーンの実装

検索結果をプロンプトに組み込んで、LLMが**根拠に基づいた回答**を生成するように設計します。

In [ ]:
# ======================================================
# 【演習 5-1】RAGチェーンの実装
# ======================================================

class MunicipalityRAGBot:
    """自治体FAQ RAGチャットボット"""

    SYSTEM_PROMPT = """
    あなたは市役所の窓口AIアシスタントです。
    提供された参考情報をもとに、住民の質問に丁寧に回答してください。

    ルール：
    1. 参考情報に記載されている内容のみを使って回答する
    2. 参考情報にない内容については「詳しくは窓口にお問い合わせください」と案内する
    3. 個人情報（氏名・住所・マイナンバー等）は絶対に聞かない
    4. 丁寧で分かりやすい言葉を使う
    5. 回答の最後に関連する窓口の情報があれば案内する
    """

    def __init__(self, vector_store: FAQVectorStore):
        self.vector_store = vector_store
        self.model = genai.GenerativeModel('gemini-1.5-flash')
        self.chat_history = []

    def answer(self, question: str) -> Tuple[str, List[str]]:
        """
        質問に対してRAGで回答を生成する
        
        Returns:
            (回答テキスト, 参照したチャンクのリスト)
        """
        # Step 1: 関連チャンクを検索
        relevant_chunks = self.vector_store.search(question, n_results=3)
        context = '\n\n'.join(relevant_chunks)

        # Step 2: プロンプトを組み立てる（RAGの核心）
        prompt = f"""{self.SYSTEM_PROMPT}

=== 参考情報（市のFAQより）===
{context}
=== 参考情報ここまで ===

住民からの質問：{question}

上記の参考情報をもとに回答してください。"""

        # Step 3: LLMで回答を生成
        response = self.model.generate_content(prompt)
        answer_text = response.text

        # 履歴に保存
        self.chat_history.append({
            'question': question,
            'answer': answer_text,
            'sources': relevant_chunks
        })

        return answer_text, relevant_chunks


# RAGボットを初期化
rag_bot = MunicipalityRAGBot(vector_store)

# テスト
print('=== RAGシステム 動作テスト ===')
test_questions = [
    '住民票をコンビニで取得するにはどうすればいいですか？',
    '保育所の申込みはいつですか？',
    '道路に大きな穴があります。どこに連絡すれば良いですか？'
]

for q in test_questions:
    print(f'\n💬 質問: {q}')
    answer, sources = rag_bot.answer(q)
    print(f'🏛️  回答: {answer}')
    print(f'📚 参照チャンク数: {len(sources)}個')

---
## Section 6：GradioでチャットUIを構築

PythonコードだけでWebチャット画面を作成します。

In [ ]:
# ======================================================
# 【演習 6-1】GradioでチャットUIを作成
# ======================================================

# 別インスタンスを用意（Gradio用）
gradio_bot = MunicipalityRAGBot(vector_store)

def chat_with_bot(message: str, history: list) -> str:
    """Gradioのチャット関数"""
    if not message.strip():
        return '質問を入力してください。'
    answer, _ = gradio_bot.answer(message)
    return answer


# Gradio UIの構築
with gr.Blocks(title='市役所窓口AI', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏛️ 市役所窓口AIアシスタント
    住民票・健康保険・介護・子育て・道路に関するご質問にお答えします。
    
    > ⚠️ このシステムは講座演習用のデモです。実際の手続きは市役所にご確認ください。
    """)

    chatbot = gr.ChatInterface(
        fn=chat_with_bot,
        examples=[
            '住民票の写しをコンビニで取得するにはどうすればいいですか？',
            '転入届はどこで手続きしますか？',
            '保育所の申込みはいつですか？',
            '道路に穴があります。どこに連絡すれば良いですか？',
            '児童手当の金額を教えてください'
        ],
        title='',
    )

# Colabで起動（share=Trueで外部URLも生成）
demo.launch(share=True)

---
## Section 7：PDFから直接読み込む（発展）

実際の自治体PDFを使いたい場合のコードです。

In [ ]:
# ======================================================
# 【発展】PDFからテキストを抽出してRAGに読み込む
# ======================================================
from google.colab import files

def extract_text_from_pdf(pdf_path: str) -> str:
    """PDFからテキストを抽出する"""
    doc = fitz.open(pdf_path)
    text = ''
    for page_num, page in enumerate(doc, 1):
        page_text = page.get_text()
        if page_text.strip():
            text += f'\n--- ページ {page_num} ---\n{page_text}'
    doc.close()
    return text


def load_pdf_to_vectorstore(pdf_path: str, vector_store: FAQVectorStore):
    """PDFを読み込んでベクトルストアに追加する"""
    print(f'📄 PDFを読み込み中: {pdf_path}')
    text = extract_text_from_pdf(pdf_path)
    print(f'   抽出文字数: {len(text)}文字')

    pdf_chunks = split_into_chunks(text, chunk_size=400)
    print(f'   チャンク数: {len(pdf_chunks)}個')

    vector_store.add_documents(pdf_chunks)
    print(f'✅ PDFのロード完了')


# 実行方法：
# 1. 以下のセルを実行するとファイルアップロードダイアログが開きます
# 2. 自治体のFAQ PDFをアップロードしてください

print('以下のコードのコメントを外してPDFをアップロードできます：')
print('''
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        load_pdf_to_vectorstore(filename, vector_store)
''')

---
## 🎯 チャレンジ課題

### 課題 A：回答の精度を評価する
以下の質問をして、回答の精度を◎/○/△/✗で評価してみてください。

| 質問 | 期待する回答のポイント | 評価 |
|------|---------------------|------|
| 住民票をコンビニで取るには？ | マイナンバーカード必要・200円 | |
| 要介護認定の手続きは？ | 申請→調査→審査→通知（約30日） | |
| 保育所の申込書類は？ | 就労証明書・申請書等 | |

### 課題 B：FAQデータを追加する
`SAMPLE_FAQ` に新しいQ&Aを追加して、ベクトルストアを再構築してください。
（例：ゴミ収集・税金・選挙 など）

### 課題 C（発展）：回答にソース表示を追加する
GradioのUIに「参照した情報源」を表示する機能を追加してください。

---

## Day 2 まとめ

| 学んだこと | ポイント |
|-----------|--------|
| チャンク分割 | 意味のある単位でテキストを分割する |
| Embedding | テキストをベクトルに変換して意味を数値で表現 |
| ベクトル検索 | 質問と意味が近い文書を高速に検索 |
| RAGプロンプト | 検索結果を文脈としてLLMに渡す |
| Gradio UI | Pythonだけで動くWebチャット画面 |

### 次のDay 3では...
今日作ったシステムをベースに、**担当業務テーマ**（土木・福祉・教育など）を選んで  
オリジナル機能を追加し、チームで発表します！